# 🏃 App 1 — Tạo index khuôn mặt & số BIB (Colab/GPU)

Notebook này quét ảnh từ một hoặc nhiều folder Google Drive, sinh **embedding khuôn mặt**
và **đọc số BIB**, rồi xuất file `index.zip` cho từng folder.

**Cách dùng:**
1. **Runtime → Change runtime type → GPU**
2. Upload file `app1_indexer.zip` (chứa code ứng dụng) khi được hỏi
3. Chạy lần lượt 3 cell → giao diện hiện ra
4. Dán link/ID folder Drive → bấm **Bắt đầu quét**
5. Mỗi folder xong → tự tải `index.zip` về máy + lưu lên Drive (tuỳ chọn)

## 1. Upload code & cài thư viện (5–7 phút lần đầu)

In [ ]:
import os, zipfile

WORK   = '/content/app1_indexer'
MARKER = '/content/.deps_installed_v4'

# --- Upload và giải nén zip code ---
if not os.path.exists(WORK):
    from google.colab import files
    print('📦 Hãy upload file app1_indexer.zip:')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall('/content')
    os.remove(zip_name)
    # Xử lý trường hợp zip có thư mục gốc hoặc không
    if not os.path.exists(WORK):
        # Tìm thư mục vừa giải nén
        extracted = [d for d in os.listdir('/content')
                     if os.path.isdir(f'/content/{d}') and d not in
                     ('sample_data', '.config', '.local')]
        for d in extracted:
            if os.path.exists(f'/content/{d}/build_index.py'):
                os.rename(f'/content/{d}', WORK)
                break
        else:
            # File nằm trực tiếp trong /content
            os.makedirs(WORK, exist_ok=True)
            for f in ('build_index.py', 'face_engine.py', 'bib_engine.py',
                      'config.py', 'drive_source.py', 'bib_detector.py'):
                if os.path.exists(f'/content/{f}'):
                    os.rename(f'/content/{f}', f'{WORK}/{f}')

os.chdir(WORK)

NEEDED = ['build_index.py', 'face_engine.py', 'bib_engine.py',
          'config.py', 'drive_source.py', 'bib_detector.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f'Thiếu file: {missing}'
print('✅ Code sẵn sàng tại:', os.getcwd())

# --- Cài deps (chỉ chạy 1 lần mỗi runtime) ---
if os.path.exists(MARKER):
    print('✅ Thư viện đã cài từ trước (skip).')
else:
    !pip install -q \
      "numpy==1.26.4" \
      insightface==0.7.3 onnxruntime-gpu==1.19.2 faiss-cpu==1.8.0 \
      paddleocr==2.7.3 paddlepaddle==2.6.1 ultralytics==8.2.0 \
      opencv-python-headless==4.10.0.84 pandas==2.2.2 pyarrow==16.1.0 \
      google-api-python-client==2.134.0 google-auth==2.30.0 \
      python-dotenv==1.0.1 tqdm==4.66.4 ipywidgets==8.1.3
    !pip install -q --force-reinstall --no-deps "numpy==1.26.4"
    open(MARKER, 'w').close()
    print('\n⚠️  Đã cài xong. KILL RUNTIME để load numpy đúng phiên bản...')
    print('    → Sau khi Colab tự reconnect, CHẠY LẠI CELL NÀY (sẽ skip cài).')
    os.kill(os.getpid(), 9)

!nvidia-smi -L
print('✅ Sẵn sàng.')

## 2. Đăng nhập Google Drive

In [ ]:
import os, sys

WORK = '/content/app1_indexer'
if os.getcwd() != WORK:
    os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)

from google.colab import auth
auth.authenticate_user()
from google.auth import default
creds, _ = default(scopes=['https://www.googleapis.com/auth/drive'])

# Import engines 1 lần
os.environ.setdefault('FACE_CTX_ID', '0')
os.environ.setdefault('ENABLE_BIB', 'true')

from drive_source import DriveSource
from config import IndexerConfig
from build_index import build_index

drive = DriveSource(credentials=creds)
print('✅ Đã đăng nhập Google Drive.')

## 3. Giao diện quét — dán link Drive và bấm chạy

In [ ]:
import re, shutil, time, traceback
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files as colab_files
from googleapiclient.http import MediaFileUpload

# ─── Helpers ───────────────────────────────────────────────────────────────────

def _extract_folder_id(text: str) -> str | None:
    """Trích folder ID từ link Drive hoặc ID thuần."""
    text = text.strip()
    if not text:
        return None
    # Dạng: https://drive.google.com/drive/folders/<ID>?...
    m = re.search(r'folders/([a-zA-Z0-9_-]+)', text)
    if m:
        return m.group(1)
    # Dạng ID thuần (không chứa khoảng trắng, >=10 ký tự)
    if re.fullmatch(r'[a-zA-Z0-9_-]{10,}', text):
        return text
    return None


def _upload_to_drive(zip_path: str, folder_name: str) -> str:
    """Upload file zip lên root Drive, trả về link."""
    service = drive._service
    file_metadata = {
        'name': f'index_{folder_name}.zip',
        'mimeType': 'application/zip',
    }
    media = MediaFileUpload(zip_path, mimetype='application/zip', resumable=True)
    uploaded = service.files().create(
        body=file_metadata, media_body=media, fields='id,webViewLink'
    ).execute()
    return uploaded.get('webViewLink', f"ID: {uploaded['id']}")


# ─── UI Widgets ────────────────────────────────────────────────────────────────

txt_links = widgets.Textarea(
    placeholder='Dán link hoặc ID folder Drive, mỗi dòng 1 folder.\n'
                'VD: https://drive.google.com/drive/folders/1ABC...\n'
                '    hoặc chỉ cần ID: 1ABC...',
    layout=widgets.Layout(width='100%', height='120px'),
)
chk_bib = widgets.Checkbox(value=True, description='Đọc số BIB (chậm hơn)')
chk_drive = widgets.Checkbox(value=True, description='Lưu index lên Drive')
chk_download = widgets.Checkbox(value=True, description='Tải index về máy')
num_limit = widgets.IntText(value=0, description='Giới hạn ảnh:', style={'description_width': '90px'},
                            layout=widgets.Layout(width='200px'))
btn_start = widgets.Button(description='🚀 Bắt đầu quét', button_style='success',
                           layout=widgets.Layout(width='200px', height='38px'))
out = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', padding='8px',
                                           max_height='500px', overflow_y='auto'))

lbl_hint = widgets.HTML('<small><i>Giới hạn ảnh = 0 nghĩa là quét toàn bộ. '
                        'Đặt 50 để chạy thử nhanh.</i></small>')

display(widgets.VBox([
    widgets.HTML('<h3>📷 Quét folder Drive</h3>'),
    txt_links,
    widgets.HBox([chk_bib, chk_download, chk_drive]),
    widgets.HBox([num_limit, lbl_hint]),
    btn_start,
    out,
]))


# ─── Logic xử lý ──────────────────────────────────────────────────────────────

def _on_start(btn):
    btn.disabled = True
    out.clear_output()
    with out:
        lines = txt_links.value.strip().splitlines()
        folder_ids = []
        for line in lines:
            fid = _extract_folder_id(line)
            if fid:
                folder_ids.append(fid)

        if not folder_ids:
            print('❌ Không tìm thấy folder ID hợp lệ. Hãy dán link hoặc ID vào ô trên.')
            btn.disabled = False
            return

        print(f'📋 Tìm thấy {len(folder_ids)} folder để quét.\n')
        limit = num_limit.value if num_limit.value > 0 else None
        enable_bib = chk_bib.value

        os.environ['ENABLE_BIB'] = 'true' if enable_bib else 'false'

        for i, fid in enumerate(folder_ids, 1):
            print(f'{"="*60}')
            print(f'▶ [{i}/{len(folder_ids)}] Folder: {fid}')
            print(f'{"="*60}')

            os.environ['DRIVE_FOLDER_ID'] = fid
            output_dir = f'/content/output_{fid[:12]}'
            os.environ['INDEX_OUTPUT_DIR'] = output_dir

            try:
                config = IndexerConfig.from_env()
                t0 = time.time()
                build_index(config, limit=limit, drive=drive)
                elapsed = time.time() - t0
                print(f'⏱️  Hoàn thành trong {elapsed:.0f}s')

                # Tạo zip
                zip_path = f'/content/index_{fid[:12]}'
                shutil.make_archive(zip_path, 'zip', output_dir)
                zip_file = zip_path + '.zip'
                size_mb = os.path.getsize(zip_file) / (1024 * 1024)
                print(f'📦 Đã tạo: {zip_file} ({size_mb:.1f} MB)')

                # Tải về máy
                if chk_download.value:
                    print('⬇️  Đang tải về máy...')
                    colab_files.download(zip_file)

                # Upload lên Drive
                if chk_drive.value:
                    print('☁️  Đang upload lên Drive...')
                    link = _upload_to_drive(zip_file, fid[:12])
                    print(f'✅ Đã upload: {link}')

                print()

            except Exception:
                traceback.print_exc()
                print(f'❌ Lỗi folder {fid}, chuyển sang folder tiếp.\n')
                continue

        print(f'\n🎉 Xong tất cả {len(folder_ids)} folder!')

    btn.disabled = False

btn_start.on_click(_on_start)

## 6. Tải log (chạy khi gặp lỗi)

Tải `run_log.txt` về thư mục `face-app/app1_indexer/` rồi nhắn "đọc log.txt" để Claude xem.

In [ ]:
from google.colab import files
files.download('/content/run_log.txt')